# Public Benchmark Baseline: Supervised Encoder From Scratch


In [ ]:
# Cell 1 - Setup & Install
import json
import os
import pickle
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml", "-q"])
    import yaml

REPO_DIR = Path("/kaggle/working/denoising-event-sequences")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_ROOT = Path("/kaggle/working/data")
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = Path("/kaggle/working/outputs/public_benchmarks")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    import torchmetrics  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchmetrics", "-q"])
    import torchmetrics  # noqa: F401

def run_logged(cmd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(REPO_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")


In [ ]:
# Shared - Prepare Public Benchmark Data If Missing
DATASET_NAME = "gender"  # "gender" or "age_group"
DATASET_CONFIG_PATHS = {
    "gender": REPO_DIR / "configs" / "datasets" / "gender.yaml",
    "age_group": REPO_DIR / "configs" / "datasets" / "age_group.yaml",
}
PROCESSED_DIR = PROCESSED_ROOT / f"{DATASET_NAME}_benchmark"
RUN_PREPARE_DATA_IF_MISSING = True
FORCE_REBUILD_PROCESSED = False
MAX_SEQ_LEN = 512

required_artifacts = ["events.parquet", "canonical_events.parquet", "transformed_events.parquet", "splits.json", "preprocessor.pkl", "prepared_config.yaml", "data_report.json"]
artifacts_exist = all((PROCESSED_DIR / name).exists() for name in required_artifacts)
if artifacts_exist:
    with (PROCESSED_DIR / "data_report.json").open() as f:
        existing_report = json.load(f)
    artifacts_exist = bool(existing_report.get("events_are_raw_pretransform", False))
if FORCE_REBUILD_PROCESSED and PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
    artifacts_exist = False

if RUN_PREPARE_DATA_IF_MISSING and not artifacts_exist:
    run_logged(
        [
            sys.executable,
            REPO_DIR / "scripts" / "prepare_public_benchmark_data.py",
            "--dataset", DATASET_NAME,
            "--raw-root", RAW_ROOT,
            "--output-dir", PROCESSED_DIR,
            "--max-seq-len", MAX_SEQ_LEN,
        ],
        OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_prepare_public_benchmark_data.log",
    )

assert all((PROCESSED_DIR / name).exists() for name in required_artifacts), PROCESSED_DIR
with (PROCESSED_DIR / "data_report.json").open() as f:
    data_report = json.load(f)
print(json.dumps(data_report, indent=2)[:3000])


In [ ]:
# Cell 4 - Supervised Encoder Configs
SUPERVISED_ENCODER_EXPERIMENTS = {
    "supervised_full": {"label_fraction": 1.0, "frozen_encoder": False},
    "supervised_low_10pct": {"label_fraction": 0.10, "frozen_encoder": False},
}
SMOKE_RUN = False

BASE_CONFIG_PATH = REPO_DIR / "configs" / "base.yaml"
DATASET_CONFIG_PATH = DATASET_CONFIG_PATHS[DATASET_NAME]
RUN_SUPERVISED_ENCODER = True


In [ ]:
# Cell 5 - Create Lightweight Runtime Overrides
CONFIG_DIR = OUTPUT_ROOT / DATASET_NAME / "supervised_encoder" / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
with BASE_CONFIG_PATH.open() as f:
    base_cfg = yaml.safe_load(f)
with DATASET_CONFIG_PATH.open() as f:
    dataset_cfg = yaml.safe_load(f)
num_classes = 2 if DATASET_NAME == "gender" else 4
selection_metric = "roc_auc" if DATASET_NAME == "gender" else "macro_f1"

def deep_update(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = deep_update(out[k], v) if isinstance(out.get(k), dict) and isinstance(v, dict) else v
    return out

runtime_base = deep_update(base_cfg, dataset_cfg)
runtime_base["data"]["timestamp_col"] = "event_timestamp"
runtime_base["data"]["max_seq_len"] = 512
runtime_base["model"]["max_seq_len"] = runtime_base["data"]["max_seq_len"]
runtime_base["training"].update({
    "num_classes": num_classes,
    "task": "binary" if num_classes == 2 else "multiclass",
    "selection_metric": selection_metric,
    "num_epochs_finetune": 4 if SMOKE_RUN else 40,
    "early_stopping_patience": 2 if SMOKE_RUN else 8,
})

runtime_config_path = CONFIG_DIR / f"{DATASET_NAME}_supervised_encoder.yaml"
with runtime_config_path.open("w") as f:
    yaml.safe_dump(runtime_base, f, sort_keys=False)
runtime_config_path


In [ ]:
# Cell 6 - Train Supervised Encoder Variants
metrics_rows = []
if RUN_SUPERVISED_ENCODER:
    for exp_name, exp_cfg in SUPERVISED_ENCODER_EXPERIMENTS.items():
        out_dir = OUTPUT_ROOT / DATASET_NAME / "supervised_encoder" / exp_name
        run_logged(
            [
                sys.executable,
                REPO_DIR / "scripts" / "finetune.py",
                "--config", runtime_config_path,
                "--data-dir", PROCESSED_DIR,
                "--output-dir", out_dir,
                "--label-fraction", exp_cfg["label_fraction"],
            ],
            OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_{exp_name}.log",
        )
        metrics_files = sorted((out_dir / "metrics").glob("*_finetune_metrics.json"))
        with metrics_files[-1].open() as f:
            metrics = json.load(f)["test_metrics"]
        metrics_rows.append({"experiment": exp_name, **metrics})

supervised_metrics_df = pd.DataFrame(metrics_rows)
supervised_metrics_df.to_csv(OUTPUT_ROOT / DATASET_NAME / "supervised_encoder_metrics.csv", index=False)
supervised_metrics_df


In [ ]:
# Cell 7 - Artifact Summary
print(OUTPUT_ROOT / DATASET_NAME / "supervised_encoder_metrics.csv")
